In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
BASE_PATH = 'data/DeepFake/Dataset'
IMAGES_PATH = os.path.join(BASE_PATH, 'faces_224')
METADATA_PATH = os.path.join(BASE_PATH, 'metadata.csv') 

# CRITICAL SETTINGS
# We use Batch Size 16 because EfficientNetB4 is memory hungry.
# If you have a powerful GPU (RTX 3090/4090), you can try 32.
BATCH_SIZE = 32
SAMPLE_SIZE = 14000  # High sample size to fix "Recall" issues
IMG_SIZE = 224

print(f"Configuration set: Batch={BATCH_SIZE}, Samples={SAMPLE_SIZE}, Model=EfficientNetB4")

In [ ]:
try:
    meta = pd.read_csv(METADATA_PATH)
except FileNotFoundError:
    meta = pd.read_json(METADATA_PATH.replace('.csv', '.json')).T

print(f"Total Metadata Rows: {len(meta)}")

# Balance the dataset (50% Real / 50% Fake)
real_df = meta[meta["label"] == "REAL"]
fake_df = meta[meta["label"] == "FAKE"]

# Sample safely
actual_sample_size = min(len(real_df), len(fake_df), SAMPLE_SIZE)
real_df = real_df.sample(actual_sample_size, random_state=42)
fake_df = fake_df.sample(actual_sample_size, random_state=42)

sample_meta = pd.concat([real_df, fake_df])
print(f"Training with balanced total: {len(sample_meta)} images")

# Split: 80% Train, 20% Test, then split Train again for Validation
Train_set, Test_set = train_test_split(sample_meta, test_size=0.2, random_state=42, stratify=sample_meta['label'])
Train_set, Val_set  = train_test_split(Train_set, test_size=0.15, random_state=42, stratify=Train_set['label'])

In [ ]:
def retrieve_dataset(set_name):
    images, labels = [], []
    for (img_name, imclass) in zip(set_name['videoname'], set_name['label']):
        # Replace extension from .mp4 to .jpg
        file_name = img_name[:-4] + '.jpg' 
        image_path = os.path.join(IMAGES_PATH, file_name)
        
        try:
            img = cv2.imread(image_path)
            if img is not None:
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                images.append(img)
                labels.append(1 if imclass == 'FAKE' else 0)
        except Exception:
            pass 
    return np.array(images), np.array(labels)

print("Loading dataset images... (Please wait)")
X_train, y_train = retrieve_dataset(Train_set)
X_val, y_val = retrieve_dataset(Val_set)
X_test, y_test = retrieve_dataset(Test_set)
print(f"Data Loaded. Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
def create_tf_dataset(X, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.map(lambda x, y: (preprocess(tf.cast(x, tf.float32)), y))
    if shuffle:
        ds = ds.shuffle(2000, seed=42)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = create_tf_dataset(X_train, y_train, shuffle=True)
val_ds = create_tf_dataset(X_val, y_val)
test_ds = create_tf_dataset(X_test, y_test)

# Robust Augmentation to reduce bias
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip(mode="horizontal", seed=42),
    tf.keras.layers.RandomRotation(factor=0.1, seed=42),
    tf.keras.layers.RandomContrast(factor=0.2, seed=42),
    tf.keras.layers.RandomZoom(height_factor=0.1, width_factor=0.1)
])

In [ ]:
print("Building EfficientNetB4 Model...")
base_model = tf.keras.applications.EfficientNetB4(
    weights="imagenet", 
    include_top=False, 
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False 

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)

# Feature Extraction Block (Brain)
x = tf.keras.layers.Dense(512, activation='relu')(x)
x = tf.keras.layers.BatchNormalization()(x) # Stabilizes training
x = tf.keras.layers.Dropout(0.5)(x)         # Prevents overfitting

outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
model = tf.keras.Model(inputs, outputs)

# Optimizer (Adam usually works better for EfficientNet than SGD)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])

In [ ]:
print("\nPhase 1: Warming up top layers...")
history = model.fit(train_ds, validation_data=val_ds, epochs=3)

In [ ]:
print("\nPhase 2: Fine-tuning EfficientNet...")

# Unfreeze the last 100 layers for deep learning
base_model.trainable = True
for layer in base_model.layers[:-100]: 
    layer.trainable = False

# Callbacks for smart training
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.2,     
    patience=2,     
    min_lr=1e-6,    
    verbose=1
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_deepfake_efficientnet.h5', 
    monitor='val_accuracy', 
    save_best_only=True, 
    mode='max', 
    verbose=1
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=5,     
    restore_best_weights=True
)

# Recompile with low learning rate
optimizer_fine = tf.keras.optimizers.Adam(learning_rate=0.0001) # Lower LR for fine-tuning
model.compile(loss="binary_crossentropy", optimizer=optimizer_fine, metrics=["accuracy"])

history_fine = model.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=20, 
    callbacks=[checkpoint, early_stop, lr_scheduler]
)

In [ ]:
print("\nLoading best model for evaluation...")
model.load_weights('best_deepfake_efficientnet.h5')

print("\nGenerating Classification Report...")
y_pred_probs = model.predict(test_ds)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

print(classification_report(y_test, y_pred, target_names=['REAL', 'FAKE']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['REAL', 'FAKE'])
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap=plt.cm.Blues, values_format='d', ax=ax)
plt.title('Confusion Matrix: EfficientNetB4')
plt.show()

print("Process Complete. Model saved as 'best_deepfake_efficientnet.h5'")